# Notebook 02: Pricing Instance Demo

In [1]:
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

try:
    from notebooks._helpers import repo_root
except ModuleNotFoundError:
    from _helpers import repo_root

ROOT = repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.problem_generator import compute_feasibility_stats
from src.classical_baselines import cp_sat_solver
from src.reduction import build_surrogate, surrogate_objective
from notebooks._helpers import load_instance, decode_config_table, format_optimal_table, format_feasibility_stats

SEED = 42
np.random.seed(SEED)

## Pricing problem in one block
- A **feature** is an optional vehicle package component (e.g., seats, roof).
- A **tier** is one of {Standard, Premium, Luxury} encoded in 2 bits (with one invalid code).
- A configuration is **feasible** if it has no invalid tier code, satisfies bundle constraints, and respects the luxury-cap rule.
- The objective $F(x)$ is expected revenue across customer segments minus penalties for infeasibility.

In [2]:
instances = [
    ('3-feature (6-bit)', 'data/instances/pricing_3feat_6bit.json'),
    ('4-feature (8-bit)', 'data/instances/pricing_4feat_8bit.json'),
    ('5-feature (10-bit)', 'data/instances/pricing_5feat_10bit.json'),
]

consolidated_rows = []

In [3]:
for instance_name, path in instances:
    prob = load_instance(path)
    stats = compute_feasibility_stats(prob)
    x_opt, f_opt = prob.brute_force_solve()

    display(Markdown(f'### {instance_name}'))

    summary = pd.DataFrame([{
        'bits': prob.n_bits,
        'feasible states': stats['valid_count'],
        'total states': stats['total_count'],
        'optimum F*': float(f_opt),
    }])
    display(summary)

    print(f'Brute-force optimum: x*={format(x_opt, f"0{prob.n_bits}b")}, F*={f_opt:.2f}')
    decoded = decode_config_table(prob, x_opt)
    display(decoded)

    display(format_feasibility_stats(stats))

    try:
        x_cp, f_cp = cp_sat_solver(prob)
        print(f'CP-SAT: x={format(x_cp, f"0{prob.n_bits}b")}, F={f_cp:.2f}, matches_bruteforce={x_cp == x_opt and abs(f_cp - f_opt) < 1e-9}')
    except Exception as exc:
        print(f'CP-SAT skipped: {exc}')

    consolidated_rows.append(format_optimal_table(instance_name, prob, x_opt))

### 3-feature (6-bit)

,bits,feasible states,total states,optimum F*
0,6,23,64,3000.0


Brute-force optimum: x*=010101, F*=3000.00


,feature,tier,price
0,Heated Seats,Premium,900.0
1,Panoramic Roof,Premium,1300.0
2,Premium Audio,Premium,800.0


,valid fraction,invalid-tier count,bundle violations,luxury-cap violations
0,0.359375,37,8,1


...

,bits,feasible states,total states,optimum F*
0,10,154,1024,3000.0


Brute-force optimum: x*=0000010000, F*=3000.00


,feature,tier,price
0,Heated Seats,Standard,500.0
1,Panoramic Roof,Standard,800.0
2,Premium Audio,Premium,800.0
3,Adaptive Cruise,Standard,600.0
4,Ambient Lighting,Standard,300.0


,valid fraction,invalid-tier count,bundle violations,luxury-cap violations
0,0.150391,781,240,106


CP-SAT: x=0000010000, F=3000.00, matches_bruteforce=True


## Pricing -> DQI surrogate hand-off (compact)
This is the exact bridge from the pricing objective $F(x)$ to the max-XORSAT-style surrogate used by DQI.

In [4]:
showcase_name, showcase_path = instances[-1]  # 5-feature frozen instance
showcase_prob = load_instance(showcase_path)
n_bits = showcase_prob.n_bits
k = min(15, (2 ** n_bits) - 1)
sur = build_surrogate(showcase_prob, k=k)
B, v, w = sur['B'], sur['v'], sur['weights']
indices, coeffs = sur['indices'], sur['coefficients']
m = sur['m']

print(f'Instance: {showcase_name}')
print(f'n={n_bits}, k={k}, m={m}')

max_rows = min(10, m)
bv_df = pd.DataFrame(B[:max_rows], columns=[f'b{j}' for j in range(n_bits)])
bv_df.insert(0, 'term', range(max_rows))
bv_df['v'] = v[:max_rows]
display(bv_df)

terms_df = pd.DataFrame({
    'rank': np.arange(1, m + 1),
    'index_S': indices,
    'mask_bits': [format(int(idx), f'0{n_bits}b') for idx in indices],
    'coeff': coeffs,
    '|coeff|': np.abs(coeffs),
})
display(terms_df.head(8))

x_opt, _ = showcase_prob.brute_force_solve()
candidate_x = [int(x_opt), 0]
fg_df = pd.DataFrame({
    'x': candidate_x,
    'bitstring': [format(x, f'0{n_bits}b') for x in candidate_x],
    'F(x)': [showcase_prob.evaluate(x) for x in candidate_x],
    'G_weighted(x)': [surrogate_objective(x, B, v, w, n_bits) for x in candidate_x],
})
fg_df

Instance: 5-feature (10-bit)
n=10, k=15, m=15


,term,b0,b1,b2,b3,b4,b5,b6,b7,b8,b9,v
0,0,0,0,0,0,0,0,1,0,0,0,0
1,1,0,0,0,0,1,0,0,0,0,0,0
2,2,0,0,1,0,0,0,0,0,0,0,0
3,3,1,0,0,0,0,0,0,0,0,0,0
4,4,0,0,0,0,0,0,0,0,1,0,0
5,5,1,1,0,0,0,0,0,0,0,0,1
6,6,0,0,1,1,0,0,0,0,0,0,1
7,7,0,0,0,0,0,0,0,0,0,1,0
8,8,0,1,0,0,0,0,0,0,0,0,0
9,9,0,0,0,1,0,0,0,0,0,0,0


,rank,index_S,mask_bits,coeff,|coeff|
0,1,8,0000001000,22081.318359,22081.318359
1,2,32,0000100000,22047.177734,22047.177734
2,3,128,0010000000,15830.830078,15830.830078
3,4,512,1000000000,15811.601562,15811.601562
4,5,2,0000000010,15795.478516,15795.478516
5,6,768,1100000000,-15412.851562,15412.851562
6,7,192,0011000000,-15397.363281,15397.363281
7,8,1,0000000001,9205.644531,9205.644531


,x,bitstring,F(x),G_weighted(x)
0,16,0000010000,3000.0,89098.623047
1,0,0000000000,2600.0,89014.892578


## Consolidated optimal configuration artifact

In [5]:
optimal_table = pd.concat(consolidated_rows, ignore_index=True)
optimal_table

,instance,feature,tier,price
0,3-feature (6-bit),Heated Seats,Premium,900.0
1,3-feature (6-bit),Panoramic Roof,Premium,1300.0
2,3-feature (6-bit),Premium Audio,Premium,800.0
3,4-feature (8-bit),Heated Seats,Standard,500.0
4,4-feature (8-bit),Panoramic Roof,Standard,800.0
5,4-feature (8-bit),Premium Audio,Standard,400.0
6,4-feature (8-bit),Adaptive Cruise,Premium,1100.0
7,5-feature (10-bit),Heated Seats,Standard,500.0
8,5-feature (10-bit),Panoramic Roof,Standard,800.0
9,5-feature (10-bit),Premium Audio,Premium,800.0


**Interpretation:** The pricing toy model is fully transparent and exactly verifiable. For each scale (6/8/10 bits), the notebook exposes feasibility structure and shows the exact globally optimal human-readable configuration.